# Lab 03: 單層感知器與活化函數 — 解決真實鳶尾花分類問題

在本單元中，我們將使用機器學習歷史上最經典的真實資料集：**安德森鳶尾花卉資料集 (Iris Dataset)**。
我們將實作一個單層感知器模型，並探索在真實世界中，什麼樣的特徵分布是「線性可分（可以用一條直線分開）」的，以及什麼時候它會失效，進而推導出為什麼我們需要活化函數與多層神經網路。

### 任務 1: 載入真實的鳶尾花資料集 (Iris Dataset)
鳶尾花資料集包含 3 種花卉（Setosa、Versicolor、Virginica）的 150 筆測量資料，特徵包括花萼與花瓣的長度與寬度。
我們使用 Scikit-learn 的 `load_iris` 來載入資料。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_iris

# 載入鳶尾花資料
iris = load_iris()
X_raw = iris.data  # 特徵: 花萼長度, 花萼寬度, 花瓣長度, 花瓣寬度
y_raw = iris.target # 標籤: 0 (Setosa), 1 (Versicolor), 2 (Virginica)
feature_names = iris.feature_names
target_names = iris.target_names

print("特徵名稱:", feature_names)
print("類別名稱:", target_names)
print(f"資料維度: {X_raw.shape}")

### 任務 2: 設計實驗 1 — 線性可分問題 (Setosa vs. Versicolor)
我們先取出 `Setosa (0)` 與 `Versicolor (1)` 兩類花卉，並且只取前兩個特徵 **「花萼長度 (Sepal Length)」** 與 **「花萼寬度 (Sepal Width)」**。
我們在二維空間中將其繪製出來，觀察這兩類花的分布。

In [ ]:
# 篩選前兩類花 (Setosa=0, Versicolor=1)
mask_1 = y_raw < 2
X_exp1 = X_raw[mask_1][:, :2]  # 只取前兩個特徵
y_exp1 = y_raw[mask_1]

# 轉為 PyTorch 張量
X_tensor_1 = torch.tensor(X_exp1, dtype=torch.float32)
y_tensor_1 = torch.tensor(y_exp1, dtype=torch.float32).unsqueeze(1)

# 繪製資料點分布
plt.figure(figsize=(7, 5))
plt.scatter(X_exp1[y_exp1==0, 0], X_exp1[y_exp1==0, 1], color='blue', label='Setosa (0)', s=80, edgecolors='k')
plt.scatter(X_exp1[y_exp1==1, 0], X_exp1[y_exp1==1, 1], color='red', label='Versicolor (1)', s=80, edgecolors='k')
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title("Experiment 1: Setosa vs. Versicolor")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("思考：從圖中看，我們能不能用『一條直線』完美地將藍點和紅點劃分開來？")

### 任務 3: 定義單層感知器模型
單層感知器包含一個線性運算層 (nn.Linear) `nn.Linear(2, 1)`（輸入特徵為 2，輸出為 1）與一個 `nn.Sigmoid` 活化函數。

In [ ]:
class Perceptron(nn.Module):
    def __init__(self):
        super(Perceptron, self).__init__()
        # TODO: 定義一個線性層，輸入維度為 2，輸出維度為 1
        # 提示：self.linear = nn.Linear(2, 1)
        # ??? (請在此填寫你的答案)
        # (已幫你完成) Sigmoid 活化函數
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # (已幫你完成) 將 x 送入線性層再通過活化函數
        out = self.sigmoid(self.linear(x))
        return out

model = Perceptron()
print(model)

### 任務 4: 訓練模型並觀察分類邊界
我們定義輔助函數來訓練並繪製綠色的分類邊界（決策邊界，Decision Boundary）。

In [ ]:
def train_and_plot(model, X_tensor, y_tensor, title, epochs=1000):
    criterion = nn.BCELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1)
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_tensor)
        loss = criterion(predictions, y_tensor)
        loss.backward()
        optimizer.step()
        
    # 繪圖
    X_np = X_tensor.numpy()
    y_np = y_tensor.squeeze().numpy()
    
    w1, w2 = model.linear.weight[0].detach().numpy()
    b = model.linear.bias[0].detach().numpy()
    
    plt.figure(figsize=(8, 6))
    colors = ['blue' if label == 0 else 'red' for label in y_np]
    plt.scatter(X_np[:, 0], X_np[:, 1], c=colors, s=80, edgecolors='k', zorder=3)
    
    # 決策邊界線: w1*x1 + w2*x2 + b = 0 => x2 = -(w1*x1 + b)/w2
    x_vals = np.linspace(X_np[:, 0].min()-0.5, X_np[:, 0].max()+0.5, 100)
    if w2 != 0:
        y_vals = -(w1 * x_vals + b) / w2
        plt.plot(x_vals, y_vals, '--', color='green', linewidth=2.5, label='Decision Boundary')
        
    # 背景填色
    xx, yy = np.meshgrid(np.linspace(X_np[:, 0].min()-0.5, X_np[:, 0].max()+0.5, 100),
                         np.linspace(X_np[:, 1].min()-0.5, X_np[:, 1].max()+0.5, 100))
    grid_t = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        pred = model(grid_t).reshape(xx.shape).numpy()
    plt.contourf(xx, yy, pred, levels=50, cmap='bwr', alpha=0.2, zorder=1)
    
    plt.title(title)
    plt.xlabel("Sepal Length (cm)")
    plt.ylabel("Sepal Width (cm)")
    plt.xlim(X_np[:, 0].min()-0.3, X_np[:, 0].max()+0.3)
    plt.ylim(X_np[:, 1].min()-0.3, X_np[:, 1].max()+0.3)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

# 開始訓練實驗一
print("=== 開始訓練實驗一：Setosa (藍) vs. Versicolor (紅) ===")
model_exp1 = Perceptron()
train_and_plot(model_exp1, X_tensor_1, y_tensor_1, "Experiment 1: Setosa vs. Versicolor (Linearly Separable)")

### 任務 5: 設計實驗 2 — 非線性可分問題 (Versicolor vs. Virginica)
現在，我們將任務變難。我們篩選出 **「Versicolor (1)」** 與 **「Virginica (2)」** 這兩類花，同樣只取前兩個特徵。
我們在二維空間中將其繪製出來，並將 Virginica 的標籤轉為 `0`，Versicolor 的標籤維持 `1` 以便進行二分類。

In [ ]:
# 篩選後兩類花 (Versicolor=1, Virginica=2)
mask_2 = y_raw >= 1
X_exp2 = X_raw[mask_2][:, :2]
y_exp2 = y_raw[mask_2]

# 為了做二分類，將標籤對應為: Virginica=0, Versicolor=1
y_exp2_binary = np.where(y_exp2 == 2, 0, 1)

# 轉為 PyTorch 張量
X_tensor_2 = torch.tensor(X_exp2, dtype=torch.float32)
y_tensor_2 = torch.tensor(y_exp2_binary, dtype=torch.float32).unsqueeze(1)

print("=== 開始訓練實驗二：Virginica (藍) vs. Versicolor (紅) ===")
model_exp2 = Perceptron()
train_and_plot(model_exp2, X_tensor_2, y_tensor_2, "Experiment 2: Versicolor vs. Virginica (Non-Linearly Separable)")

print("思考：在實驗二中，紅點與藍點重疊在了一起。單個神經元畫出的『直線』還能完美區分它們嗎？")

### 重點省思與討論
*   **線性可分限制**：單個感知器在數學上等同於一條「直線」。當真實世界的資料（如實驗二的鳶尾花）分布重疊或呈現複雜非線性時，直線劃分會產生大量的誤判。
*   這就是為什麼我們需要 **多層感知器 (MLP)** — 利用多個神經元與活化函數組合，在空間中揉捏出「彎曲的非線性決策邊界」，才能解決複雜的真實世界分類難題！
*   在**東吳大學資料科學系**的課程中，我們會學到各種處理非線性分類的進階技巧，包括 SVM、Random Forest、以及今日最火熱的深度神經網路！

---

## Lab 03 學習重點小結
### 核心概念掌握
- 感知器 (Perceptron) 是最基礎的人工神經元，可解決線性可分問題。
- 活化函數 (Activation Function) 決定神經元的輸出，引入非線性。
- XOR 問題無法用單層感知器解決，因為它線性不可分。
- Sigmoid 函數將輸出壓縮至 (0,1)，適合二分類問題的輸出層。

### 快速自評測驗

**Q1. 下列哪個問題無法用單一感知器解決？**
- A. AND 邏輯閘
- B. OR 邏輯閘
- C. XOR 邏輯閘

<details><summary>查看解答</summary>

**答案：C — XOR 需要多層網路才能解決，因其資料點線性不可分。**

</details>

**Q2. ReLU 活化函數的特性為？**
- A. 輸出值介於 0~1
- B. 負數輸入輸出 0，正數輸入輸出原值
- C. 輸出值固定為 ±1

<details><summary>查看解答</summary>

**答案：B — ReLU(x) = max(0, x)，計算簡單且有效緩解梯度消失問題。**

</details>

### 完成確認清單
在繼續下一個 Lab 前，請確認你能做到：
- [ ] 我可以用一句話解釋「感知器 (Perceptron) 是最基礎的人工神經元，可解」
- [ ] 我可以用一句話解釋「活化函數 (Activation Function) 決定神」
- [ ] 我可以用一句話解釋「XOR 問題無法用單層感知器解決，因為它線性不可分。」
- [ ] 我的程式碼成功執行並得到預期輸出
